In [1]:
import sys
from pathlib import Path

project_root = Path(
    r"C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation"
)
module_dir = str(project_root / "tools" / "scientific_code" / "PPREstimation")

if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

import pandas as pd
import numpy as np
from PPRCalculator import PPRCalculator
from ModelData import ModelData

# Loading

In [2]:
model_path = (
    "../models/"
    "52_1_Sea_of_Okhotsk_NE_(1980)/"
    "model.json"
)

model = ModelData(
    model_path,
    model_name="Sea of Okhotsk NE",
    model_year="1980",
)

# Inspect or edit these before constructing the calculator:
# model.groups_data['biomass_accum'] = np.nan
# model.groups_data
# model.DC
# model.det_fate

In [33]:
calculator = PPRCalculator.from_modeldata(
    modeldata=model,
    underdetermined=True,     # Estimate missing flows using LIM if True
    zero_catch=True,           # Replace missing catch with zero
    zero_biomass_accum=False,   # Replace missing biomass accumulation with zero
    default_gs=True,           # Supply GS=0.2 for regular groups missing GS
    weight_flow=1.0,           # LIM minimum-flow penalty weight
    weight_guess=1.0,          # LIM guess-deviation penalty weight
    normalize_DC=True,         # Normalize consumer diet sums if True
    DC_tol=0.001,             # Allowed deviation of diet sums from 1
    balance_BA_after_DC_normalization=False
)

C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\ModelData.py:663: RuntimeWarning: 4 consumer group(s) have a diet composition (including diet_import) that does not sum to 1 (tol=0.001): 18 (Bathylagidae)=0.9990, 9 (Salmon)=0.9990, 6 (Walleye pollock)=1.0010, 2 (Toothed whales)=0.9980.
  warnings.warn(f"{len(bad)} consumer group(s) have a diet composition (including diet_import) "


# DC

In [12]:
model.DC.sum(axis=1).rename(index=model.seq2name).round(decimals=7)
# model.DC.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7).columns
# model.DC.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7)

diet_import                 0.0
Detritus                    0.0
Marine mammals              1.0
Sharks                      1.0
Small yellow croakers       1.0
Large yellow croakers       1.0
Omnivores                   1.0
Benthivores/piscivores      1.0
Planktivores/piscivores     1.0
Planktivores/Benthivores    1.0
Bombay duck                 1.0
Hairtails                   1.0
Piscivores                  1.0
Benthivores                 1.0
Planktivores                1.0
Cephalopods                 1.0
Shrimps                     1.0
Crabs                       1.0
Other invertebrates         1.0
Echinoderms                 1.0
Benthic Crustaceans         1.0
Mollusks                    1.0
Polychaetes                 1.0
Zooplankton                 1.0
Phytoplankton               0.0
dtype: float64

In [7]:
calculator.get_DC().sum(axis=1).round(decimals=3)

39    0.0
38    0.0
37    1.0
36    1.0
35    1.0
34    1.0
33    1.0
32    1.0
31    1.0
30    1.0
29    1.0
28    1.0
27    1.0
26    1.0
25    1.0
24    1.0
23    1.0
22    1.0
21    1.0
20    1.0
19    1.0
18    1.0
17    1.0
16    1.0
15    1.0
14    1.0
13    1.0
12    1.0
11    1.0
10    1.0
9     1.0
8     1.0
7     1.0
6     1.0
5     1.0
4     1.0
3     1.0
2     0.0
1     0.0
dtype: float64

# Groups Data

In [30]:
model.groups_data.round(decimals=7)[['group_name', 'biomass', 'biomass_accum']]

,group_name,biomass,biomass_accum
group_seq,,,
29,Detritus,NaN,NaN
28,Phytoplankton,160.943,NaN
27,Phytobenthos,12.579,NaN
26,Herbivorous zooplankton,197.480,NaN
25,Predatory zooplankton,86.792,NaN
24,2nd level benthos,143.052,NaN
23,Carnivorous invertebrates,12.948,NaN
22,Jellyfish,0.980,NaN
21,Squids,1.541,NaN


In [31]:
calculator.get_groups_df().round(decimals=7)[['group_name', 'biomass', 'biomass_accum']]

,group_name,biomass,biomass_accum
group_seq,,,
30,diet_import,1.000,0.000000
29,Detritus,1.000,21997.696125
28,Phytoplankton,160.943,9.120485
27,Phytobenthos,12.579,0.042541
26,Herbivorous zooplankton,197.480,17.429786
25,Predatory zooplankton,86.792,-15.461188
24,2nd level benthos,143.052,0.288498
23,Carnivorous invertebrates,12.948,0.159808
22,Jellyfish,0.980,0.003562


In [26]:
# model.groups_data.round(decimals=7)[['group_name', 'ge', 'ee']]

In [27]:
# calculator.get_groups_df().round(decimals=7)[['group_name', 'ge', 'ee', 'tl']]

# Is Model Balanced

In [21]:
balanced, p, q = calculator.is_model_balanced()
balanced

True

In [22]:
(100 * (p - calculator.p)/p).rename(index=calculator.seq2name)

group_seq
diet_import                          NaN
Detritus                    0.000000e+00
Marine mammals              0.000000e+00
Sharks                      0.000000e+00
Small yellow croakers       0.000000e+00
Large yellow croakers       0.000000e+00
Omnivores                  -1.542532e-14
Benthivores/piscivores      0.000000e+00
Planktivores/piscivores     0.000000e+00
Planktivores/Benthivores    0.000000e+00
Bombay duck                 0.000000e+00
Hairtails                   0.000000e+00
Piscivores                  0.000000e+00
Benthivores                 0.000000e+00
Planktivores                0.000000e+00
Cephalopods                 0.000000e+00
Shrimps                     0.000000e+00
Crabs                       0.000000e+00
Other invertebrates         0.000000e+00
Echinoderms                 0.000000e+00
Benthic Crustaceans         0.000000e+00
Mollusks                    0.000000e+00
Polychaetes                 0.000000e+00
Zooplankton                 0.000000e+00
Phytop

In [8]:
calc_balanced = calculator.balanced_model
print(calc_balanced.n_balance_runs)
balanced, p, q = calc_balanced.is_model_balanced()
100 * (p - calc_balanced.p)/p

1


group_seq
25             NaN
24    0.000000e+00
23    0.000000e+00
22    0.000000e+00
21    0.000000e+00
20    0.000000e+00
19    0.000000e+00
18    0.000000e+00
17    0.000000e+00
16    0.000000e+00
15    0.000000e+00
14   -1.731950e-14
13    0.000000e+00
12    0.000000e+00
11    0.000000e+00
10    0.000000e+00
9    -2.107821e-14
8     0.000000e+00
7     0.000000e+00
6     0.000000e+00
5     0.000000e+00
4    -2.132190e-14
3     0.000000e+00
2     0.000000e+00
1     0.000000e+00
dtype: float64

# Diagnostics

In [22]:
result, sppr, A, L = calculator.diagnose_sppr(
    TE_option='GE',
    return_sppr=True
)

In [8]:
result, sppr, A, L = calculator.diagnose_sppr(
    TE_option='TE',
    return_sppr=True
)

C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\PPRCalculator.py:2706: RuntimeWarning: 2 group(s) have TE=0 because EE=0 (M0=p): 5 (Seabirds), 3 (Sperm whales). Their consumed PP is re-credited to detritus (fix_EE_0_cases=True) so the TE SPPR stays PP-balanced.
  warnings.warn(
C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\PPRCalculator.py:2731: RuntimeWarning: 1 group(s) have near-zero TE (0 < EE << 1): 1 (Baleen whales). Their SPPR is near-singular (inherent to the TE method); the detritus SPPR may be inflated and is not corrected by fix_EE_0_cases.
  warnings.warn(


In [23]:
result['model_input']

{'status': 'WARN',
 'is_model_balanced': True,
 'p_max_rel_residual': 1.3552527156068805e-16,
 'q_max_rel_residual': 1.4354398702224247e-16,
 'dc_rows_sum_to_1': True,
 'dc_max_deviation': 3.3306690738754696e-16,
 'n_negative_catch': 0,
 'n_zero_catch': 30,
 'total_catch': 0.0,
 'has_catch': False,
 'has_ee_issues': True,
 'n_ee0': 2,
 'n_ee_marginal': 0,
 'n_ee_gt_1': 0,
 'ee0_groups': [5, 3],
 'ee_marginal_groups': []}

In [24]:
result['divergence']

{'status': 'WARN',
 'solve_error': None,
 'b': 0.152043528728438,
 'b_converges': True,
 'rho_living': 0.8671658823517999,
 'living_converges': True,
 'sppr_det': {29: 1.251013685876604},
 'max_sppr_det': 1.251013685876604,
 'max_sppr_group': {'seq': 2,
  'tl': 4.4522396470093515,
  'sppr': 2083179.0167024508,
  'inv_te': 655.5},
 'max_tl_group': {'seq': 3,
  'tl': 4.903229690175536,
  'sppr': 58254.218718140255,
  'inv_te': 227.5},
 'n_negative_sources': 0,
 'expect_negatives': False,
 'near_singular_te': []}

In [19]:
result['balance']

{'status': 'OK',
 'is_balanced': True,
 'inflow': 26654.193900000002,
 'outflow': 26654.130786912032,
 'rel_gap': 2.3678483096096615e-06}

In [20]:
result['footprint']

{'ppr_all': 0.0,
 'ppr_inner': 0.0,
 'ppr_pp_only': 0.0,
 'npp': 26654.193900000002,
 'ppr2npp': 0.0,
 'ppr2npp_pp_only': 0.0}

In [28]:
result['config']

{'TE_option': 'GE',
 'det_open_mode': 'none',
 'det_theta': 1.0,
 'det_external_sppr': 0.0,
 'det_collapse_mode': 'never',
 'explicit_TE': False,
 'method': 'single_detritus',
 'would_pool': False,
 'model': 'East China Sea (2018)'}

In [14]:
result['warnings']

['total catch is 0: every footprint number is identically 0 (the divergence diagnostics are unaffected)',
 "2 group(s) with EE=0 (all production is non-predatory death): 5 (Seabirds), 3 (Sperm whales). Under TE_option='TE' their TE row is 0, which severs them from the nullspace and leaks the PP they consumed",
 'rho(A_LL)=0.8672 is within 0.3 of divergence (max SPPR 2.08e+06 at group 2 (Toothed whales), TL 4.45)']

In [14]:
(sppr<0).any().any()

np.False_